# Loading Environment Variables

Before interacting with any LLM provider, we need to securely load our environment variables.

Instead of hardcoding sensitive information such as API keys inside the source code, we store them in a `.env` file and load them at runtime using the `python-dotenv` package.

This approach provides several benefits:

- Keeps API keys out of the source code.
- Prevents accidentally exposing credentials when sharing the project.
- Makes it easy to switch between different environments (development, testing, production).
- Follows industry best practices for secret management.

Calling `load_dotenv()` loads every variable defined in the `.env` file into the current Python environment, allowing libraries like LangChain to automatically retrieve the required credentials.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## No memory

# Building an Agent Without Memory

In this example, we create a basic AI Agent using LangChain without attaching any memory component.

The agent is initialized using the `create_agent()` function and powered by the `gpt-5-nano` model. We then send a single `HumanMessage` containing the user's introduction.

```python
Hello my name is Seán and my favourite colour is green.
```

The message is passed to the agent using the `invoke()` method.

Unlike sending plain strings, LangChain represents conversations using message objects such as:

- `HumanMessage`
- `AIMessage`
- `SystemMessage`
- `ToolMessage`

This standardized message format enables LangChain to manage conversations, tools, and agent workflows in a structured way.

At this stage, the agent has **no persistent memory**. It only processes the messages provided in the current request and generates a response based solely on that context.

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent= create_agent(
    "gpt-5-nano"
)

question= HumanMessage(content="Hello my name is Seán and my favourite colour is green")

response= agent.invoke({"messages":[question]})


# Inspecting the Agent's Response

Instead of printing only the generated text, we inspect the complete response object returned by the agent.

The response contains much more than the final answer. It includes the full conversation state represented as a list of messages.

Typically, this list contains:

- The original `HumanMessage`
- The generated `AIMessage`

Examining the complete response is useful for understanding how LangChain internally manages conversations and becomes especially valuable when working with:

- Tool Calling
- Multi-step Agent Workflows
- Memory
- LangGraph
- Debugging complex applications

Rather than treating the model as a simple text generator, LangChain exposes the conversation as structured data that can be inspected, modified, or reused throughout the application.

In [3]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='73c54272-3a95-434a-a68e-f0f68bcae9d4'),
              AIMessage(content='Nice to meet you, Seán! Green is a great color—calming, natural, and versatile.\n\nWhat would you like to do today? Here are a few quick options:\n- Learn a quick Gaelic greeting or phrase\n- Explore green color ideas (shades, palettes, combos)\n- Get a fun fact about the color green in nature\n- Write a short poem or tiny story featuring Seán and the color green\n- Talk about eco-friendly tips or sustainability\n\nTell me what you’re in the mood for, or just share something else you’d like to chat about.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 895, 'prompt_tokens': 18, 'total_tokens': 913, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_token

# Demonstrating Stateless Behavior

Although the user previously introduced themselves, we now ask a follow-up question:

```python
What is my name and what is my favourite colour?
```

Since this request only contains the new message, the agent has no access to the earlier conversation.

Large Language Models are **stateless by default**, meaning they do not automatically remember previous interactions.

Every request is processed independently unless the previous conversation history is explicitly provided or a memory mechanism is configured.

As a result, the agent cannot answer questions about earlier interactions because that information is no longer available in the current context.

In [5]:
question2= HumanMessage(content="What is my name and what is my favourite colour?")

response= agent.invoke({"messages":[question2]})

pprint(response)

{'messages': [HumanMessage(content='What is my name and what is my favourite colour?', additional_kwargs={}, response_metadata={}, id='0f7f3b02-aadc-40fc-8228-d4c9f3fefef1'),
              AIMessage(content='I don’t know your name or your favourite colour yet — you haven’t shared them. If you tell me, I can refer to you by your name and use your colour in this chat. Want to give me a name and a colour? Or should I pick a default for now?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 643, 'prompt_tokens': 17, 'total_tokens': 660, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DpfruE7AAtsKASCKquDURKvmYVNUm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': N

## Memory

# Adding Persistent Memory

To enable the agent to remember previous interactions, we attach a memory component using LangGraph's `InMemorySaver`.

```python
checkpointer = InMemorySaver()
```

The checkpointer automatically stores and retrieves the conversation history between requests.

Instead of manually sending the entire conversation every time, LangGraph reconstructs the conversation using the stored state.

The memory component is passed directly to the agent during initialization:

```python
create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver()
)
```

This transforms the agent from a stateless conversational model into a stateful agent capable of maintaining context across multiple interactions.

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

agent= create_agent(
    model= "gpt-5-nano",
    checkpointer= InMemorySaver()
)

question= HumanMessage(content="Hello my name is Seán and my favourite colour is green")

config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [question]},
    config,  
)
pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='79efeddb-6910-4b7f-bac2-8035072e3b8d'),
              AIMessage(content='Nice to meet you, Seán! Green is a great color—calm and refreshing. What would you like to talk about today? If you want, I can tailor the chat around green topics (nature, eco-friendly ideas, green recipes) or we can chat about anything you like.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 708, 'prompt_tokens': 18, 'total_tokens': 726, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dpg2EaxOhGjtFlhqwv9R7Do5tvbf0', 'service_tier': 'default', 'finish_reason': '

# Reusing the Same Conversation

We now ask another question using the **same** `thread_id`.

Although the new request only contains:

```python
What's my favourite colour?
```

the agent successfully answers because LangGraph automatically restores the previous conversation from memory.

Behind the scenes, the framework combines:

- Previously stored messages
- The current user message

before sending the request to the language model.

From the model's perspective, it receives the complete conversation history even though we only provided the latest message in our code.

In [10]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config
)
pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='79efeddb-6910-4b7f-bac2-8035072e3b8d'),
              AIMessage(content='Nice to meet you, Seán! Green is a great color—calm and refreshing. What would you like to talk about today? If you want, I can tailor the chat around green topics (nature, eco-friendly ideas, green recipes) or we can chat about anything you like.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 708, 'prompt_tokens': 18, 'total_tokens': 726, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dpg2EaxOhGjtFlhqwv9R7Do5tvbf0', 'service_tier': 'default', 'finish_reason': '

# Key Takeaways

This notebook demonstrates one of the most important concepts in modern AI agent development.

Without memory:

- Every request is completely independent.
- Previous conversations are forgotten.
- Developers must manually provide conversation history.

With memory enabled:

- Conversation history is automatically persisted.
- Context is restored transparently.
- The agent can maintain long-running conversations.
- Different users can be isolated using unique `thread_id`s.

Understanding the distinction between stateless language models and stateful AI agents is fundamental when building production-ready conversational applications.